# 04 — Classificazione Lamfalussy a 4 Livelli + Heatmap di Ibridità

Questo notebook classifica gli articoli secondo la tassonomia canonica Lamfalussy a 4 livelli (Commissione Europea, 2001 → riforma post-crisi 2010).

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Testo articolo + full doc | LLM: classifica provision per provision in L1–L4 | `segments_lamfalussy.csv` |
| **B** | Classificazioni per articolo | Calcolo entropia normalizzata per articolo | `nodes_lamfalussy.csv` |
| **C** | Entropia per articolo | Aggregazione → score ibridità per atto | `nodes_hybridity.csv` |
| **D** | CSV di output | Build `heatmaps.json` + patch HTML | file aggiornati |

## 0. Configurazione

**Modifica solo questa cella.**

In [ ]:
MATERIA_NAME = "appalti_it"

# ── Modello ────────────────────────────────────────────────────────────────────
LLM_MODEL          = "gpt-4.1-mini"   # modello da usare per la classificazione

# ── Parametri API ─────────────────────────────────────────────────────────────
LLM_MAX_TOKENS     = 2000   # token risposta (JSON con provisions può essere lungo)
LLM_DELAY_SECONDS  = 0.3
LLM_MAX_RETRIES    = 3
LLM_RETRY_DELAY    = 5.0

# ── Parallelismo ──────────────────────────────────────────────────────────────
MAX_WORKERS        = 5      # thread paralleli per le chiamate API

# ── Checkpoint ────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 50     # articoli tra un salvataggio e il successivo

# ── Contesto documento (cap per non sforare il context window) ────────────────
DOC_CONTEXT_MAX_CHARS = 40_000   # caratteri massimi del documento di contesto

# ── Visualizzazione HTML ───────────────────────────────────────────────────────
# True  = appalti_it e domini con HTML; False = domini senza HTML (es. fdi_screening)
ENABLE_HTML_OUTPUT = True

## 1. Import e Percorsi

In [ ]:
import os, re, json, math, time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Override via variabile d'ambiente (per esecuzione programmatica) ──────────
_materia_override = os.environ.get('MATERIA_OVERRIDE', '')
if _materia_override:
    MATERIA_NAME = _materia_override
_html_override = os.environ.get('ENABLE_HTML_OVERRIDE', '')
if _html_override:
    ENABLE_HTML_OUTPUT = _html_override.lower() in ('true', '1', 'yes')

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

# Auto-detect file nodi: prova prima nodes_texts_it.csv (appalti), poi nodes_texts.csv
_it_path  = os.path.join(output_path, 'nodes_texts_it.csv')
_all_path = os.path.join(output_path, 'nodes_texts.csv')
INPUT_FILE = _it_path if os.path.exists(_it_path) else _all_path

# Auto-detect file nodi EU (cerca nodes_texts_eu_*.csv, altrimenti nessun file EU)
import glob as _glob_nb04
_eu_candidates = _glob_nb04.glob(os.path.join(output_path, 'nodes_texts_eu_*.csv'))
INPUT_FILE_EU   = _eu_candidates[0] if _eu_candidates else ''

SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
NODES_DIFFUSION_FILE    = os.path.join(output_path, 'nodes_diffusion.csv')

# File HTML per la visualizzazione (None se ENABLE_HTML_OUTPUT=False)
HTML_FILE = os.path.join('..', f'{MATERIA_NAME}_network.html') if ENABLE_HTML_OUTPUT else None

# Se HTML_FILE e' impostato e non esiste ancora, lo crea dal template
if HTML_FILE and not os.path.exists(HTML_FILE):
    _template = os.path.join('..', 'network_template.html')
    if os.path.exists(_template):
        import shutil as _shutil_setup
        _shutil_setup.copy(_template, HTML_FILE)
        print(f'Creato {HTML_FILE} dal template')

# Prompt template esterno — modifica questo file per cambiare il prompt
PROMPT_FILE = os.path.join('..', 'notebooks', 'prompt.txt')

# ── Costanti Lamfalussy ────────────────────────────────────────────────────────
LAMFALUSSY_LEVELS = [
    {'key': 'L1', 'name': 'Level 1 — Framework principles', 'label': 'level_1'},
    {'key': 'L2', 'name': 'Level 2 — Operational rules',    'label': 'level_2'},
    {'key': 'L3', 'name': 'Level 3 — Supervisory convergence', 'label': 'level_3'},
    {'key': 'L4', 'name': 'Level 4 — Enforcement',          'label': 'level_4'},
]

LAMF_KEYS = [l['key']   for l in LAMFALUSSY_LEVELS]
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]
LABEL_TO_KEY = {l['label']: l['key'] for l in LAMFALUSSY_LEVELS}

print(f"Materia:    {MATERIA_NAME}")
print(f"Modello:    {LLM_MODEL}")
print(f"Livelli:    {LAMF_KEYS}")
print(f"Input IT:   {INPUT_FILE}  (esiste: {os.path.exists(INPUT_FILE)})")
print(f"Input EU:   {INPUT_FILE_EU or '(nessuno)'}")
print(f"HTML:       {HTML_FILE or '(disabilitato)'}")

## 2. Caricamento Dati e Segmentazione

In [ ]:
# ── Carica IT ────────────────────────────────────────────────────────────────
nodes_it = pd.read_csv(INPUT_FILE)

if os.path.exists(INPUT_FILE_EU):
    nodes_eu_raw = pd.read_csv(INPUT_FILE_EU)
    # Normalizza colonne EU allo schema IT atteso da questo notebook
    nodes_eu = nodes_eu_raw.copy()
    if 'Id'    not in nodes_eu.columns: nodes_eu['Id']    = nodes_eu.get('celex', nodes_eu.get('label', ''))
    if 'Label' not in nodes_eu.columns: nodes_eu['Label'] = nodes_eu.get('celex', nodes_eu.get('label', ''))
    if 'text_status' not in nodes_eu.columns:
        nodes_eu['text_status'] = nodes_eu['segments'].apply(
            lambda s: 'ok' if (pd.notna(s) and str(s).strip() not in ('', '[]')) else 'no_text'
        )
    nodes = pd.concat([nodes_it, nodes_eu], ignore_index=True)
    print(f'IT: {len(nodes_it)} nodi  |  EU: {len(nodes_eu)} nodi  |  Totale: {len(nodes)}')
else:
    nodes = nodes_it
    print(f'File EU non trovato ({INPUT_FILE_EU}) — solo IT: {len(nodes)} nodi')

nodes_ok = nodes[nodes['text_status'] == 'ok'].copy()

print(f"Nodi totali: {len(nodes)}  |  ok: {len(nodes_ok)}")
print(nodes['text_status'].value_counts().to_string())

# ── Esplode segmenti in DataFrame flat ────────────────────────────────────────
rows = []
for _, node in nodes_ok.iterrows():
    celex = str(node.get('Label', node['Id']))
    title = str(node.get('title', ''))
    raw   = node.get('segments', '')
    if pd.isna(raw) or not str(raw).strip():
        continue
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        continue
    seen = set()
    for i, s in enumerate(segs):
        if s.get('tipo') != 'articolo':
            continue
        testo = str(s.get('testo', '')).strip()
        if len(testo) < 30:
            continue
        idf = str(s.get('identificatore', i))
        seg_id = f"{celex}__{idf}"
        if seg_id in seen:
            continue
        seen.add(seg_id)
        rows.append({
            'segment_id':    seg_id,
            'celex':         celex,
            'node_id':       str(node['Id']),
            'title_atto':    title,
            'tipo':          s.get('tipo'),
            'identificatore': idf,
            'testo':         testo,
        })

articles_df = pd.DataFrame(rows)

# ── Mappa celex → full_text (per il contesto documento) ──────────────────────
FULL_TEXTS = {}
for _, node in nodes_ok.iterrows():
    celex = str(node.get('Label', node['Id']))
    ft    = str(node.get('full_text', '') or '')
    if ft and ft != 'nan':
        FULL_TEXTS[celex] = ft

print(f"\nArticoli estratti: {len(articles_df):,}")
print(f"Atti coinvolti:    {articles_df['celex'].nunique():,}")
print(f"Full text disponibili: {len(FULL_TEXTS):,}")

## 3. Fase A — Classificazione Lamfalussy per Articolo (LLM)

Per ogni articolo:
1. Il testo viene tokenizzato (whitespace split) e ogni token riceve un indice 1-based
2. L'LLM segmenta il testo in **provisions** contigue e classifica ciascuna come
   `level_1`, `level_2`, `level_3`, `level_4` o `unassigned`
3. Le percentuali L1–L4 emergono dai **conteggi di token** per livello

Il documento integrale è fornito come contesto (`[FULL DOCUMENT]`) per permettere
all'LLM di interpretare ogni articolo nel suo contesto normativo.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROMPT TEMPLATE — letto da file esterno
# ─────────────────────────────────────────────────────────────────────────────
with open(PROMPT_FILE, encoding='utf-8') as _f:
    PROMPT_TEMPLATE = _f.read()

print(f'Prompt caricato da: {PROMPT_FILE}  ({len(PROMPT_TEMPLATE)} caratteri)')
for ph in ['{{DOCUMENT_CONTEXT}}', '{{CHUNK_TEXT}}', '{{EXPECTED_UNITS}}', '{{DOCUMENT_NAME}}']:
    status = '✓' if ph in PROMPT_TEMPLATE else '✗ MANCANTE'
    print(f'  {status}  {ph}')


def is_non_substantive(text: str, art_id: str = None) -> bool:
    """
    Ritorna True se l'articolo non ha contenuto normativo sostanziale.
    Rimuove lo scaffold Normattiva e gli avvisi di abrogazione tra doppie parentesi,
    poi verifica se il residuo è vuoto/punteggiatura.
    Per le sezioni AGGIORNAMENTO conserva solo quelle che citano esplicitamente
    questo specifico articolo (per non escludere disposizioni transitorie dedicate).
    """
    t = text
    # 1. Rimuovi scaffold Normattiva
    t = re.sub(r'Approfondimenti e Funzioni', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'articolo\s+precedente\s+articolo\s+successivo', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'articolo\s+(?:precedente|successivo)', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'Testo in vigore dal:\s*[\d\-\/]+', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r"aggiornamenti all.articolo", ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\bArt\.\s*\d+[\w\-]*\b', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\bArticolo\s+\d+[\w\-]*\b', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\bArticoli\b', ' ', t, flags=re.IGNORECASE)
    # 2. Rimuovi avvisi di abrogazione/soppressione tra doppie parentesi
    t = re.sub(r'\(\(\s*(?:PROVVEDIMENTO\s+)?ARTICOLO\s+ABROGAT[OA].*?\)\)', ' ', t,
               flags=re.IGNORECASE | re.DOTALL)
    t = re.sub(r'\(\(\s*PROVVEDIMENTO\s+ABROGAT[OA].*?\)\)', ' ', t,
               flags=re.IGNORECASE | re.DOTALL)
    t = re.sub(r'\(\(\s*SOPPRESS[OAE].*?\)\)', ' ', t, flags=re.IGNORECASE | re.DOTALL)
    t = re.sub(r'\(\(\s*ABROGAT[OA].*?\)\)', ' ', t, flags=re.IGNORECASE | re.DOTALL)
    # 3. Rimuovi marcatori di nota a piè di pagina ((N))
    t = re.sub(r'\(\(\d+\)\)', ' ', t)
    # 4. Gestisci sezioni AGGIORNAMENTO: conserva solo quelle che citano questo articolo
    if art_id:
        art_num = re.search(r'(\d+)', str(art_id))
        if art_num:
            art_num = art_num.group(1)
            aggiorns = re.findall(
                r'-{3,}.*?AGGIORNAMENTO\s*\(\d+\)(.*?)(?=-{3,}|$)', t,
                re.DOTALL | re.IGNORECASE
            )
            new_t = re.sub(r'-{3,}.*?AGGIORNAMENTO\s*\(\d+\)[\s\S]*', ' ', t,
                           flags=re.IGNORECASE)
            for body in aggiorns:
                if re.search(r'\b' + re.escape(art_num) + r'\b', body):
                    new_t += ' ' + body
            t = new_t
    # 5. Verifica se il residuo ha contenuto sostanziale (lettere)
    residual = re.sub(r'[\s\-\.,:;()\[\]\/\|\d]', '', t)
    return len(residual) < 20


def tokenize_with_indices(text: str) -> tuple[list[str], str]:
    """Restituisce (words, testo_indicizzato)."""
    words = text.split()
    indexed = ' '.join(f'({i+1}){w}' for i, w in enumerate(words))
    return words, indexed


def build_prompt(doc_text: str, art_text: str, art_id: str, doc_name: str) -> str:
    """Costruisce il prompt per un singolo articolo."""
    _, indexed_art = tokenize_with_indices(art_text)
    chunk = f"[ARTICLE {art_id}]\n[ARTICLE_TEXT]\n{indexed_art}"
    unit_id   = 'i000001'
    expected  = json.dumps({unit_id: {'article_id': str(art_id), 'comma_id': None}}, indent=2)
    ctx       = doc_text[:DOC_CONTEXT_MAX_CHARS] if doc_text else '(not available)'
    return (PROMPT_TEMPLATE
        .replace('{{DOCUMENT_CONTEXT}}', ctx)
        .replace('{{CHUNK_TEXT}}',       chunk)
        .replace('{{EXPECTED_UNITS}}',   expected)
        .replace('{{DOCUMENT_NAME}}',    doc_name))


def parse_response(response_text: str, art_text: str) -> dict | None:
    """
    Parsa la risposta JSON dell'LLM e calcola le percentuali L1–L4
    dal conteggio dei token classificati.
    Restituisce None in caso di errore di parsing.
    Se tutti i token sono 'unassigned', restituisce status='unassigned'
    con pcts a zero (articolo conservato nel corpus ma marcato).
    """
    try:
        clean = re.sub(r'^```[a-z]*\n?', '', response_text.strip())
        clean = re.sub(r'\n?```$', '', clean)
        data  = json.loads(clean)
    except json.JSONDecodeError:
        return None

    classes = data.get('classifications', {})
    if not classes:
        return None
    unit = next(iter(classes.values()), {})
    provisions = unit.get('provisions', [])
    if not provisions:
        return None

    words, _ = tokenize_with_indices(art_text)
    total_tokens = len(words)

    counts = {k: 0 for k in LAMF_KEYS}
    counts['unassigned'] = 0
    evidence = []

    for p in provisions:
        start  = int(p.get('start', 1))
        end    = int(p.get('end', total_tokens))
        label  = str(p.get('label', 'unassigned'))
        reason = str(p.get('reason', ''))
        span   = max(0, end - start + 1)
        key    = LABEL_TO_KEY.get(label, 'unassigned')
        counts[key] = counts.get(key, 0) + span

        if key != 'unassigned' and span > 0:
            quote = ' '.join(words[start-1:end])
            if len(quote) > 200:
                quote = quote[:197] + '...'
            evidence.append({'testo': quote, 'layer': key, 'motivo': reason})

    total_labeled = sum(counts.get(k, 0) for k in LAMF_KEYS)
    if total_labeled == 0:
        # L'LLM ha classificato tutto come unassigned — conserva con status dedicato
        return {
            'pcts': {k: 0.0 for k in LAMF_KEYS},
            'evidence': [],
            'provisions': provisions,
            'status': 'unassigned',
        }

    pcts = {k: round(counts.get(k, 0) / total_labeled * 100, 2) for k in LAMF_KEYS}
    return {'pcts': pcts, 'evidence': evidence, 'provisions': provisions, 'status': 'ok'}


def call_llm(client, prompt: str) -> tuple[str, str]:
    """Chiama l'LLM con retry. Restituisce (response_text, status)."""
    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=LLM_MAX_TOKENS,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
            )
            return resp.choices[0].message.content, 'ok'
        except openai.RateLimitError:
            time.sleep(LLM_RETRY_DELAY * (attempt + 1))
        except Exception as e:
            if attempt == LLM_MAX_RETRIES - 1:
                return str(e), 'error'
            time.sleep(LLM_RETRY_DELAY)
    return 'max_retries_exceeded', 'error'


print('Funzioni Fase A definite.')
print(f'Livelli Lamfalussy: {LAMF_KEYS}')


In [ ]:
%%time
client = OpenAI()

# ── Gestione checkpoint ────────────────────────────────────────────────────────
done_ids = set()
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    ckpt_df  = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    done_ids = set(ckpt_df['segment_id'])
    print(f'Checkpoint: {len(done_ids):,} articoli già classificati.')
else:
    ckpt_df = pd.DataFrame()
    print('Nessun checkpoint — si parte da zero.')

todo_df = articles_df[~articles_df['segment_id'].isin(done_ids)].copy()
print(f'Articoli da classificare: {len(todo_df):,}  |  già ok: {len(done_ids):,}')

if len(todo_df) == 0:
    print('✓ Tutti i segmenti già classificati — si può passare alla Fase B.')

# ── Strutture condivise tra thread ────────────────────────────────────────────
ckpt_lock   = Lock()
buffer      = []
n_ok        = 0
n_unsub     = 0
n_error     = 0
n_processed = 0


def process_article(seg: pd.Series) -> dict:
    """Classifica un articolo e restituisce la riga da salvare."""
    celex  = seg['celex']
    art_id = seg['identificatore']
    testo  = seg['testo']

    # Pre-filtro deterministico: articoli senza contenuto normativo sostanziale
    if is_non_substantive(testo, art_id):
        row = {
            'segment_id':     seg['segment_id'],
            'celex':          celex,
            'node_id':        seg['node_id'],
            'tipo':           seg['tipo'],
            'identificatore': art_id,
            'llm_status':     'unassigned',
            'evidence':       '[]',
            'provisions_raw': '[]',
        }
        for col in LAMF_COLS:
            row[col] = 0.0
        return row

    prompt = build_prompt(
        doc_text  = FULL_TEXTS.get(celex, ''),
        art_text  = testo,
        art_id    = art_id,
        doc_name  = celex,
    )
    time.sleep(LLM_DELAY_SECONDS)
    response_text, status = call_llm(client, prompt)

    parsed = None
    if status == 'ok':
        parsed = parse_response(response_text, testo)
        if parsed is None:
            status = 'parse_error'
        else:
            status = parsed.get('status', 'ok')   # 'ok' or 'unassigned'

    row = {
        'segment_id':    seg['segment_id'],
        'celex':         celex,
        'node_id':       seg['node_id'],
        'tipo':          seg['tipo'],
        'identificatore': art_id,
        'llm_status':    status,
        'evidence':      json.dumps(parsed['evidence'], ensure_ascii=False) if parsed else '[]',
        'provisions_raw': json.dumps(parsed['provisions'], ensure_ascii=False) if parsed else '[]',
    }
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        row[col] = round(parsed['pcts'][key], 2) if parsed else 0.0
    return row


def save_checkpoint(new_rows):
    if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
        existing = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    else:
        existing = pd.DataFrame()
    parts    = [p for p in [existing, pd.DataFrame(new_rows)] if not p.empty]
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    combined.to_csv(SEGMENTS_LAMF_CKPT_FILE, index=False)


# ── Loop parallelo ─────────────────────────────────────────────────────────────
total = len(todo_df)
todo_records = [row for _, row in todo_df.iterrows()]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_article, seg): seg['segment_id'] for seg in todo_records}
    for future in as_completed(futures):
        row = future.result()
        with ckpt_lock:
            buffer.append(row)
            n_processed += 1
            st = row['llm_status']
            if st == 'ok':
                n_ok += 1
            elif st == 'unassigned':
                n_unsub += 1
            else:
                n_error += 1
            if n_processed % CHECKPOINT_EVERY == 0 or n_processed == total:
                save_checkpoint(buffer)
                buffer.clear()
                pct = n_processed / total * 100 if total else 100
                print(f'  [{n_processed:>5}/{total}]  {pct:5.1f}%   ok: {n_ok}   unassigned: {n_unsub}   errori: {n_error}')

print()
print('=' * 60)
print(f'FASE A — ok: {n_ok:,}   unassigned: {n_unsub:,}   errori: {n_error:,}')
print('=' * 60)


In [ ]:
# ── Salva output finale Fase A ─────────────────────────────────────────────────
segments_lamf = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
segments_lamf.to_csv(SEGMENTS_LAMF_FILE, index=False)

ok_mask = segments_lamf['llm_status'] == 'ok'
print(f'Salvato: {SEGMENTS_LAMF_FILE}')
print(f'Totale: {len(segments_lamf):,}  |  ok: {ok_mask.sum():,}')
print()
print('Distribuzione media % per livello Lamfalussy (articoli ok):')
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = segments_lamf.loc[ok_mask, col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

### Reaggregation of split articles (FDI)

When article-length limits create fragments such as `12_p1` and `12_p2`, the cached provision labels are reassembled into one row per legal article before act-level scoring. This step is deterministic and makes no API calls.


In [ ]:
# Reaggregate cached fragments before the splitting notebook.
import sys as _sys_reagg
from pathlib import Path as _Path_reagg

_repo_root = _Path_reagg('..').resolve()
if str(_repo_root) not in _sys_reagg.path:
    _sys_reagg.path.insert(0, str(_repo_root))

from scripts.analysis_core import (
    LEVEL_COLUMNS as _LEVEL_COLUMNS,
    aggregate_acts as _aggregate_acts,
    deterministic_partition as _deterministic_partition,
    diffuse as _diffuse,
    load_edges as _load_edges,
    reaggregate_fragments as _reaggregate_fragments,
)

_cached_segments = pd.read_csv(SEGMENTS_LAMF_FILE)
_has_fragments = _cached_segments['identificatore'].astype(str).str.contains(r'_p\d+$').any()

if _has_fragments:
    _article_rows = _reaggregate_fragments(_cached_segments)

    _title_lookup = {}
    _id_column = next((c for c in ['Id', 'id', 'celex', 'Label'] if c in nodes.columns), None)
    _title_column = next((c for c in ['title', 'titolo', 'Label'] if c in nodes.columns), None)
    if _id_column and _title_column:
        _title_lookup = {
            str(row[_id_column]): str(row[_title_column])
            for _, row in nodes[[_id_column, _title_column]].dropna().iterrows()
        }

    _act_rows = _aggregate_acts(_article_rows, _title_lookup)
    _edge_rows = _load_edges(_Path_reagg(output_path))
    _diffusion_rows, _ = _diffuse(_act_rows[['celex', *_LEVEL_COLUMNS]], _edge_rows)
    _diffusion_rows = _diffusion_rows[['celex', 'H_global', 'H_local']].rename(
        columns={'H_local': 'H_local_diffusion'}
    )
    _act_rows = _act_rows.merge(_diffusion_rows, on='celex', how='left')

    _split_rows = []
    _accepted_after = {}
    for _act in _act_rows.itertuples(index=False):
        _celex = str(_act.celex)
        _before = float(_act.hybridity_score)
        _eligible = _before >= 0.35 and int(_act.n_articles) >= 8
        _partition = None
        if _eligible:
            _partition = _deterministic_partition(
                _article_rows.loc[_article_rows['celex'].astype(str).eq(_celex)],
                min_block=4,
                min_improvement=0.30,
            )
        _row = {
            'celex': _celex,
            'hybridity_score_before': round(_before, 4),
            'n_articles_reali': int(_act.n_articles),
            'eligible': _eligible,
            'outcome': 'accepted' if _partition else ('rejected' if _eligible else 'not_eligible'),
            'H_before': round(_before, 4),
            'H_after': round(float(_partition['H_after']), 4) if _partition else None,
            'delta_H': round(float(_partition['delta_H']), 4) if _partition else None,
            'H_improvement': round(float(_partition['H_improvement']), 4) if _partition else None,
            'n_blocks': int(_partition['n_blocks']) if _partition else None,
        }
        if _partition:
            _accepted_after[_celex] = float(_partition['H_after'])
        _split_rows.append(_row)

    _before_mean = float(_act_rows['hybridity_score'].mean())
    _after_mean = float(np.mean([
        _accepted_after.get(str(_act.celex), float(_act.hybridity_score))
        for _act in _act_rows.itertuples(index=False)
    ]))
    _split_rows.append({
        'celex': '__SUMMARY__',
        'hybridity_score_before': round(_before_mean, 4),
        'n_articles_reali': int(_act_rows['n_articles'].sum()),
        'eligible': None,
        'outcome': 'network_summary',
        'H_before': round(_before_mean, 4),
        'H_after': round(_after_mean, 4),
        'delta_H': round(_before_mean - _after_mean, 4),
        'H_improvement': round((_before_mean - _after_mean) / _before_mean, 4),
        'n_blocks': len(_accepted_after),
    })

    _reaggregated_dir = _Path_reagg(output_path) / 'reaggregated'
    _reaggregated_dir.mkdir(parents=True, exist_ok=True)
    _article_rows.to_csv(_reaggregated_dir / 'nodes_lamfalussy_byarticle.csv', index=False)
    _act_rows.to_csv(_reaggregated_dir / 'nodes_hybridity_byarticle.csv', index=False)
    pd.DataFrame(_split_rows).to_csv(
        _reaggregated_dir / 'splitting_byarticle.csv', index=False
    )
    print(
        f'Reaggregated {len(_cached_segments):,} fragments into '
        f'{len(_article_rows):,} articles; paired act-level H '
        f'{_before_mean:.4f} -> {_after_mean:.4f}.'
    )
else:
    print('No split-article identifiers found; reaggregation is not required.')


## 4. Fase B — Entropia per Articolo e per Atto

Per ogni articolo: **entropia di Shannon normalizzata** sulla distribuzione L1–L4.

$$H(a) = -\frac{\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)}{\log_2(4)}$$

Zero = articolo monofunzionale. Uno = distribuzione uniforme sui 4 livelli.

In [ ]:
def entropy_norm(row, lamf_cols):
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in lamf_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return np.nan
    probs = probs / s
    rawH  = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(lamf_cols)) if len(lamf_cols) > 1 else 1.0
    return float(np.clip(rawH / maxH, 0, 1))


# Ricarica dal file finale
seg_df = pd.read_csv(SEGMENTS_LAMF_FILE)
_valid_seg_ids = set(articles_df['segment_id'])

# Articoli sostanziali (classificati dall'LLM con successo)
art_df = seg_df[
    (seg_df['llm_status'] == 'ok') &
    (seg_df['segment_id'].isin(_valid_seg_ids))
].copy()

# Articoli unassigned (non sostanziali, da includere nel corpus ma con vals=0)
uns_df = seg_df[
    (seg_df['llm_status'] == 'unassigned') &
    (seg_df['segment_id'].isin(_valid_seg_ids))
].copy()

for col in LAMF_COLS:
    if col not in art_df.columns:
        art_df[col] = 0.0
    if col not in uns_df.columns:
        uns_df[col] = 0.0

# Calcola entropia solo per gli articoli sostanziali
art_df['entropy']      = art_df.apply(lambda r: entropy_norm(r, LAMF_COLS), axis=1)
art_df['dominant_lamf'] = art_df[LAMF_COLS].idxmax(axis=1).str.replace('lamf_', '', regex=False)
art_df['articolo_id']  = art_df['identificatore']

# Articoli unassigned: entropia = 0, dominant_lamf = 'unassigned'
uns_df['entropy']       = 0.0
uns_df['dominant_lamf'] = 'unassigned'
uns_df['articolo_id']   = uns_df['identificatore']

# Salva nodes_lamfalussy con entrambi (sostanziali + unassigned)
nodes_lamf_all = pd.concat([art_df, uns_df], ignore_index=True)
nodes_lamf_all.to_csv(NODES_LAMFALUSSY_FILE, index=False)

print(f'Salvato: {NODES_LAMFALUSSY_FILE}')
print(f'Articoli sostanziali: {len(art_df):,}  |  Unassigned: {len(uns_df):,}  |  Totale: {len(nodes_lamf_all):,}')
print(f'Atti coinvolti: {nodes_lamf_all["celex"].nunique():,}')
print(f'Entropia media (sostanziali): {art_df["entropy"].mean():.4f}  |  max: {art_df["entropy"].max():.4f}')
print()
print('Distribuzione media % per livello Lamfalussy (articoli sostanziali):')
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = art_df[col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")


## 5. Fase C — Score di Ibridità per Atto

Lo score di ibridità di un atto è l'**entropia della distribuzione L1-L4 media** dei suoi articoli.
Un atto è **puro** se la sua distribuzione aggregata è concentrata su un solo livello (H ≈ 0).
Un atto è **ibrido** se la sua distribuzione aggregata è equamente spalmata sui livelli (H → 1).

> **Nota:** questa misura differisce dalla media delle entropie per articolo.
> Un atto con metà articoli puri L1 e metà puri L2 ottiene H_atto ≈ 0.5 (corretto),
> mentre la media delle entropie per articolo darebbe ≈ 0 (sbagliato).

In [ ]:
LAMF_NAMES = {l['key']: l['name'] for l in LAMFALUSSY_LEVELS}

def entropy_of_mean(group, lamf_cols):
    """Entropy of the mean L1-L4 distribution across all articles in the group."""
    eps = 1e-9
    mean_vals = group[lamf_cols].mean().values.astype(float)
    s = mean_vals.sum()
    if s < eps:
        return 0.0
    probs = mean_vals / s
    rawH = -np.sum(probs * np.log2(probs + eps))
    maxH = math.log2(len(lamf_cols))
    return float(np.clip(rawH / maxH, 0, 1))


def agg_atto(group):
    # Separa sostanziali da unassigned
    sub = group[group['dominant_lamf'] != 'unassigned']
    n_unassigned = int((group['dominant_lamf'] == 'unassigned').sum())

    if len(sub) == 0:
        # Atto interamente non sostanziale
        return pd.Series({
            'hybridity_score':     np.nan,
            'hybridity_score_old': np.nan,
            'hybridity_std':       np.nan,
            'hybridity_max':       np.nan,
            'n_articles':          0,
            'n_unassigned':        n_unassigned,
            'dominant_lamf':       '',
            'dom':                 'unassigned',
            'dominant_lamf_pct':   0.0,
            'most_hybrid_article': '',
            **{col: 0.0 for col in LAMF_COLS},
        })

    dom = sub['dominant_lamf'].mode()
    hyb_max_row = sub.nlargest(1, 'entropy')
    dominant_key = dom.iloc[0] if len(dom) else ''
    return pd.Series({
        'hybridity_score':     entropy_of_mean(sub, LAMF_COLS),
        'hybridity_score_old': sub['entropy'].mean(),
        'hybridity_std':       sub['entropy'].std(),
        'hybridity_max':       sub['entropy'].max(),
        'n_articles':          len(sub),
        'n_unassigned':        n_unassigned,
        'dominant_lamf':       dominant_key,
        'dom':                 LAMF_NAMES.get(dominant_key, dominant_key),
        'dominant_lamf_pct':   (sub['dominant_lamf'] == dominant_key).mean() * 100 if dominant_key else 0.0,
        'most_hybrid_article': hyb_max_row['articolo_id'].iloc[0] if len(hyb_max_row) else '',
        **{col: sub[col].mean() for col in LAMF_COLS},
    })


# Carica nodes_lamfalussy (include sia sostanziali che unassigned)
_lamf_full = pd.read_csv(NODES_LAMFALUSSY_FILE)
hybridity_df = _lamf_full.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols  = [c for c in ['Id', 'Label', 'title'] if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
if 'Label' in nodes_meta.columns:
    nodes_meta = nodes_meta.rename(columns={'Label': 'celex'})

hybridity_df = (hybridity_df
    .merge(nodes_meta, on='celex', how='left')
    .drop_duplicates(subset=['celex'], keep='first')
    .sort_values('hybridity_score', ascending=False))

hybridity_df = hybridity_df.loc[:, ~hybridity_df.columns.duplicated()]
hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f'Salvato: {NODES_HYBRIDITY_FILE}')
print(f'Atti analizzati: {len(hybridity_df):,}')
print(f'Articoli sostanziali totali: {hybridity_df["n_articles"].sum():,}')
print(f'Articoli unassigned totali:  {hybridity_df["n_unassigned"].sum():,}')
scored_hybridity = hybridity_df[hybridity_df['n_articles'] > 0]
desc = scored_hybridity['hybridity_score'].describe()
print(f"\nStatistiche hybridity_score (solo articoli sostanziali):")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")
print()
print('Top 5 atti più ibridi (esclusi unassigned):')
for _, row in scored_hybridity.head(5).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {celex_label:<22}  score={row['hybridity_score']:.3f}  "
          f"n={row['n_articles']}  n_uns={row['n_unassigned']}  dominant={row['dominant_lamf']}")

print()
print(f'Atti con unassigned > 0:')
for _, row in hybridity_df[hybridity_df['n_unassigned'] > 0].iterrows():
    print(f"  {row['celex']:<24}  n_sub={row['n_articles']:>4}  n_uns={row['n_unassigned']:>4}  "
          f"H={row['hybridity_score']:.3f}  L1={row['lamf_L1']:.1f}%")

# Verifica CSV
_check = pd.read_csv(NODES_HYBRIDITY_FILE)
print(f"\nColonne: {_check.columns.tolist()}")


## 6. Fase D — Export heatmaps.json + Patch HTML

Costruisce `heatmaps.json` con le percentuali L1–L4 per ogni articolo, 
poi patcha l'HTML con le percentuali a 4 livelli Lamfalussy.

In [ ]:
import json as _json
import re as _re
import os as _os

# ── 1. Costruisci HEATMAPS dict ───────────────────────────────────────────────
# Mappa celex → testi articoli
ART_TEXTS = {}
if _os.path.exists(INPUT_FILE):
    texts_df = pd.read_csv(INPUT_FILE)
    id_col   = 'Id' if 'Id' in texts_df.columns else 'celex'
    for _, row in texts_df.iterrows():
        celex = str(row[id_col])
        raw   = row.get('segments', '')
        if not raw or str(raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in _json.loads(str(raw)):
                if seg.get('tipo') == 'articolo':
                    ART_TEXTS[(celex, str(seg.get('identificatore', '')))] = seg.get('testo', '')
        except Exception:
            pass

# EU texts
if INPUT_FILE_EU and _os.path.exists(INPUT_FILE_EU):
    _eu_texts = pd.read_csv(INPUT_FILE_EU)
    for _, row in _eu_texts.iterrows():
        celex = str(row.get('Label', row.get('celex', '')))
        raw   = row.get('segments', '')
        if not raw or str(raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in _json.loads(str(raw)):
                if seg.get('tipo') == 'articolo':
                    ART_TEXTS[(celex, str(seg.get('identificatore', '')))] = seg.get('testo', '')
        except Exception:
            pass

print(f'ART_TEXTS: {len(ART_TEXTS)} articoli')

# Mappa segment_id → evidence
EVIDENCE = {}
seg_file = pd.read_csv(SEGMENTS_LAMF_FILE)
for _, row in seg_file[seg_file['llm_status'] == 'ok'].iterrows():
    celex = str(row['celex'])
    idf   = str(row['identificatore'])
    raw   = row.get('evidence', '[]')
    try:
        EVIDENCE[(celex, idf)] = _json.loads(raw) if isinstance(raw, str) else []
    except Exception:
        EVIDENCE[(celex, idf)] = []
print(f'EVIDENCE:  {len(EVIDENCE)} articoli con evidence')

# Provisions token-indexed (start/end) per highlighting preciso nel pannello articolo
PROVISIONS = {}
_LBL = {'level_1':'L1','level_2':'L2','level_3':'L3','level_4':'L4','unassigned':''}
def _nlbl(l):
    return _LBL.get((l or '').lower(), (l or '').upper())

for _, row in seg_file[seg_file['llm_status'] == 'ok'].iterrows():
    celex = str(row['celex'])
    idf   = str(row['identificatore'])
    raw_p = row.get('provisions_raw', '[]')
    try:
        pvs = _json.loads(raw_p) if isinstance(raw_p, str) else []
        PROVISIONS[(celex, idf)] = [
            {'start': int(p['start']), 'end': int(p['end']),
             'label': _nlbl(p.get('label','')), 'reason': p.get('reason','')}
            for p in pvs if p.get('start') is not None and p.get('end') is not None
        ]
    except Exception:
        PROVISIONS[(celex, idf)] = []
print(f'PROVISIONS: {len(PROVISIONS)} articoli con provisions token-indexed')

# Costruisce HEATMAPS — include articoli unassigned con vals=None
HEATMAPS = {}
lamf_all = pd.read_csv(NODES_LAMFALUSSY_FILE)

for celex, grp in lamf_all.groupby('celex'):
    arts = []
    for _, row in grp.sort_values('identificatore', key=lambda s: s.apply(
        lambda x: int(_re.match(r'(\d+)', str(x)).group(1)) if _re.match(r'\d', str(x)) else 9999
    )).iterrows():
        idf            = str(row['articolo_id'])
        is_unassigned  = str(row.get('dominant_lamf', '')) == 'unassigned'

        if is_unassigned:
            arts.append({
                'id':         idf,
                'vals':       None,   # null in JSON → articolo non sostanziale
                'H':          0.0,
                'lamf':       {k: 0 for k in LAMF_KEYS},
                'H_lamf':     0.0,
                'ev':         [],
                'txt':        ART_TEXTS.get((celex, idf), ''),
                'unassigned': True,
            })
        else:
            vals  = [round(float(row.get(c, 0)), 2) for c in LAMF_COLS]
            total = sum(vals)
            if total > 0 and abs(total - 100) > 0.5:
                vals = [round(v / total * 100, 2) for v in vals]
            H_val = round(float(row.get('entropy', 0)), 4)
            lamf  = {k: vals[i] for i, k in enumerate(LAMF_KEYS)}
            arts.append({
                'id':         idf,
                'vals':       vals,
                'H':          H_val,
                'lamf':       lamf,
                'H_lamf':     H_val,
                'ev':         EVIDENCE.get((celex, idf), []),
                'provisions': PROVISIONS.get((celex, idf), []),
                'txt':        ART_TEXTS.get((celex, idf), ''),
            })
    if arts:
        HEATMAPS[celex] = arts

n_uns_hm = sum(1 for arts in HEATMAPS.values() for a in arts if a.get('unassigned'))
print(f'HEATMAPS:  {len(HEATMAPS)} atti  |  '
      f'{sum(len(v) for v in HEATMAPS.values())} articoli totali  |  '
      f'{n_uns_hm} unassigned')

# Salva heatmaps.json nella cartella di output del dominio
json_path = _os.path.join(output_path, 'heatmaps.json')
with open(json_path, 'w', encoding='utf-8') as f:
    _json.dump(HEATMAPS, f, ensure_ascii=False)
print(f'\n✓ Salvato {json_path}  ({_os.path.getsize(json_path)//1024} KB)')

if HTML_FILE:
    import shutil as _shutil_hm
    _html_dir  = _os.path.dirname(_os.path.abspath(HTML_FILE))
    _html_json = _os.path.join(_html_dir, 'heatmaps.json')
    if _os.path.abspath(json_path) != _html_json:
        _shutil_hm.copy(json_path, _html_json)
        print(f'✓ Copiato anche in {_html_json}')


In [ ]:
# ── Estrazione archi da citazioni testuali ─────────────────────────────────────
# Patterna sui riferimenti normativi italiani nel testo degli articoli
# e li risolve contro i nodi del catalogo tramite anno+numero.

import re as _re_cit

EDGES_FILE_IT = os.path.join(output_path, 'edges_it_internal.csv')

# Mappa (tipo_normalizzato, anno, numero) → slug
# costruita dai metadati seed in nodes_texts_it.csv
nodes_it_df = pd.read_csv(INPUT_FILE)
id_col_it   = 'Id' if 'Id' in nodes_it_df.columns else 'id'

TIPO_ALIASES = {
    'decreto legislativo':                     'dlgs',
    'decreto-legislativo':                     'dlgs',
    'd.lgs':                                   'dlgs',
    'd.lgs.':                                  'dlgs',
    'dlgs':                                    'dlgs',
    'legge':                                   'l',
    'l.':                                      'l',
    'decreto del presidente della repubblica': 'dpr',
    'decreto.del.presidente.della.repubblica': 'dpr',
    'd.p.r':                                   'dpr',
    'd.p.r.':                                  'dpr',
    'dpr':                                     'dpr',
    'decreto-legge':                           'dl',
    'decreto legge':                           'dl',
    'd.l.':                                    'dl',
    'dl':                                      'dl',
    'decreto ministeriale':                    'dm',
    'decreto del ministro':                    'dm',
    'd.m.':                                    'dm',
    'dm':                                      'dm',
}

MESI = r'(?:gennaio|febbraio|marzo|aprile|maggio|giugno|luglio|agosto|settembre|ottobre|novembre|dicembre)'
PAT_FULL = _re_cit.compile(
    r'(decreto(?:\s+del\s+presidente\s+della\s+repubblica|[\s\-]?legislativo|[\s\-]?legge|[\s\s]ministeriale)?'
    r'|legge|l\.)\s+'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*,?\s*n\.?\s*(\d+)',
    _re_cit.IGNORECASE
)
PAT_SHORT = _re_cit.compile(
    r'(d\.lgs\.|d\.p\.r\.|d\.l\.|d\.m\.|legge)\s*'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*[,\s]+n\.?\s*(\d+)',
    _re_cit.IGNORECASE
)

WINDOW = 120

def classify_relation(context):
    ctx = context.lower()
    if any(p in ctx for p in ['abrogat', 'abrogazion', 'cessa di avere efficacia',
        'è abrogato', 'sono abrogati', 'è abrogata', 'sono abrogate']):
        return 'REPEALS', 'mod'
    if any(p in ctx for p in ['modificat', 'sostituit', 'è sostituito', 'sono sostituiti',
        'le parole', 'dopo le parole', 'novellat', 'è così modificat']):
        return 'AMENDS', 'mod'
    if any(p in ctx for p in ['in attuazione', 'in applicazione', 'ai sensi',
        'in conformità', 'su delega', 'recepimento', 'recepisce']):
        return 'BASED_ON', 'hier'
    if any(p in ctx for p in ['deroga', 'in deroga', 'non si applica']):
        return 'DEROGATES', 'proc'
    return 'CITES', 'ref'

def normalize_tipo(raw):
    raw = raw.strip().lower().rstrip('.')
    for alias, norm in TIPO_ALIASES.items():
        if raw.startswith(alias):
            return norm
    return None

slug_lookup = {}
slug_to_year = {}

for _, row in nodes_it_df.iterrows():
    slug = str(row[id_col_it])
    parts = slug.split('_')
    if len(parts) >= 3:
        tipo, num, anno = parts[0], parts[1], parts[2]
        slug_lookup[(tipo, anno, num)] = slug
        try:
            slug_to_year[slug] = int(anno)
        except ValueError:
            pass

print(f'Lookup nodi: {len(slug_lookup)} voci')

text_edges = []
seen_text  = set()

for _, node_row in nodes_it_df.iterrows():
    src_slug = str(node_row[id_col_it])
    segs_raw = node_row.get('segments', '')
    full_txt = str(node_row.get('full_text', '') or '')

    texts_to_scan = [full_txt]
    if segs_raw and str(segs_raw) not in ('nan', '[]', ''):
        try:
            for seg in json.loads(str(segs_raw)):
                texts_to_scan.append(str(seg.get('testo', '')))
        except Exception:
            pass

    combined = ' '.join(texts_to_scan)

    for pat in [PAT_FULL, PAT_SHORT]:
        for m in pat.finditer(combined):
            tipo_raw = m.group(1)
            anno     = m.group(2)
            numero   = m.group(3)
            tipo_n   = normalize_tipo(tipo_raw)
            if not tipo_n:
                continue
            dst_slug = slug_lookup.get((tipo_n, anno, numero))
            if not dst_slug or dst_slug == src_slug:
                continue
            key = (src_slug, dst_slug)
            if key in seen_text:
                continue
            seen_text.add(key)
            start   = max(0, m.start() - WINDOW)
            end     = min(len(combined), m.end() + WINDOW)
            context = combined[start:end]
            etype, family = classify_relation(context)
            text_edges.append({
                'src_slug': src_slug,
                'dst_slug': dst_slug,
                'type':     etype,
                'family':   family,
                'w':        2 if etype in ('REPEALS','AMENDS','BASED_ON') else 1,
            })

print(f'\nArchi estratti dai testi: {len(text_edges)}')
df_text_edges = pd.DataFrame(text_edges) if text_edges else pd.DataFrame(
    columns=['src_slug', 'dst_slug', 'type', 'family', 'w'])

# Salva sempre (anche con 0 righe, così i notebook a valle non falliscono)
df_text_edges.to_csv(EDGES_FILE_IT, index=False)
print(f'Salvato: {EDGES_FILE_IT}  ({len(df_text_edges)} archi)')

if len(df_text_edges) > 0:
    print(df_text_edges['src_slug'].value_counts().head(10).to_string())

# Verifica
_echeck = pd.read_csv(EDGES_FILE_IT)
print(f'\nVerifica CSV — colonne: {_echeck.columns.tolist()}')
print(f'Righe: {len(_echeck)}')

## 7. Fase E — Entropia Globale via Diffusione di Citazioni

L'entropia locale `H` calcolata in Fase B misura solo l'**ibridità interna** di un atto:
quanto i suoi articoli sono distribuiti sui 4 livelli Lamfalussy. Non cattura la
**contaminazione funzionale via citazioni**: un atto monofunzionale che cita atti
ibridi eredita complessità nel sistema in cui opera.

Implementiamo la **diffusione del profilo di livello**:

$$
\mathbf{p}_{\text{global}}(v) = (1-\lambda)\,\mathbf{p}_{\text{local}}(v) + \lambda \sum_{u} \tilde{w}(v,u)\,\mathbf{p}_{\text{global}}(u)
$$

In forma chiusa: $\mathbf{P}_{\text{glob}} = (1-\lambda)(I - \lambda \tilde{W})^{-1} \mathbf{P}_{\text{loc}}$

con $\tilde{W}$ riga-stocastica costruita dagli archi del grafo. Gli archi
`IMPLEMENTED_BY` (EU→IT) sono **invertiti** nel calcolo perché semanticamente
l'atto IT eredita complessità interpretativa dalla direttiva EU che recepisce.

Definiamo tre quantità:
- `H_local(v)` = Shannon di $\mathbf{p}_{\text{local}}(v)$ — già calcolata in Fase B
- `H_global(v)` = Shannon di $\mathbf{p}_{\text{global}}(v)$
- `Δ(v) = H_global(v) − H_local(v)` = **deriva di contaminazione**

Δ > 0 → l'atto sembra coeso ma partecipa a un ecosistema ibrido.  
Δ < 0 → cita atti più ordinati di sé.


In [ ]:
import numpy as _np_diff
import json as _json_diff
import re as _re_diff

LAMBDA = 0.5  # peso delle citazioni nella diffusione

def _parse_js_var(html, varname):
    """Robustly extract a JS array/object using JSONDecoder.raw_decode,
    so arbitrary content inside strings (including ]; ) never truncates early."""
    m = _re_diff.search(rf'const {varname}\s*=\s*', html)
    if not m:
        raise ValueError(f'const {varname} not found in HTML')
    obj, end = _json_diff.JSONDecoder().raw_decode(html, m.end())
    return obj, m.start(), m.end(), end

def _replace_js_var(html, varname, new_obj):
    """Replace a JS array/object variable in html with new_obj serialised as JSON."""
    _, decl_start, _, json_end = _parse_js_var(html, varname)
    new_decl = f'const {varname}  = ' + _json_diff.dumps(new_obj, ensure_ascii=False) + ';'
    tail_start = json_end
    while tail_start < len(html) and html[tail_start] in ' \t':
        tail_start += 1
    if tail_start < len(html) and html[tail_start] == ';':
        tail_start += 1
    return html[:decl_start] + new_decl + html[tail_start:]

if HTML_FILE and os.path.exists(HTML_FILE):
    # Percorso HTML (appalti_it): legge NODES/EDGES dall'HTML
    with open(HTML_FILE, 'r', encoding='utf-8') as f:
        _html_d = f.read()

    _nodes_d, *_ = _parse_js_var(_html_d, 'NODES')
    _edges_d, *_ = _parse_js_var(_html_d, 'EDGES')

    if not _nodes_d:
        # HTML ancora template vuoto: costruisce NODES da nodes_hybridity.csv
        print('NODES vuoto nel HTML — costruzione nodi da nodes_hybridity.csv...')
        _hyb_build = pd.read_csv(NODES_HYBRIDITY_FILE)
        for _, _rb in _hyb_build.iterrows():
            _cx = str(_rb['celex'])
            _lf = {k: round(float(_rb.get(f'lamf_{k}', 0) or 0), 1) for k in LAMF_KEYS}
            _nodes_d.append({
                'id':           _cx,
                'H':            round(float(_rb.get('hybridity_score', 0) or 0), 3),
                'n':            int(_rb.get('n_articles', 0) or 0),
                'n_unassigned': int(_rb.get('n_unassigned', 0) or 0),
                'dom':          str(_rb.get('dom', '')),
                'dominant_lamf': str(_rb.get('dominant_lamf', '')),
                'lamf':         _lf,
                'title':        str(_rb.get('title', _cx)),
                'eu':           _cx[:1].isdigit(),
            })
        print(f'  {len(_nodes_d)} nodi IT costruiti da CSV')

    if not _edges_d:
        _efile = os.path.join(output_path, 'edges_it_internal.csv')
        if os.path.exists(_efile):
            _edf_b = pd.read_csv(_efile)
            for _, _er in _edf_b.iterrows():
                _edges_d.append({
                    's':      str(_er['src_slug']),
                    't':      str(_er['dst_slug']),
                    'family': str(_er.get('family', 'ref')),
                    'w':      float(_er.get('w', 1)),
                })
            print(f'  {len(_edges_d)} archi da edges_it_internal.csv')

    # Ensure EU→IT recep edges are always present
    _eu_it_file = os.path.join(output_path, 'edges_eu_it.csv')
    if os.path.exists(_eu_it_file):
        _existing_pairs = {(e['s'], e['t']) for e in _edges_d}
        _eu_edf = pd.read_csv(_eu_it_file)
        _added_eu = 0
        for _, _er in _eu_edf.iterrows():
            _s, _t = str(_er['src_slug']), str(_er['dst_slug'])
            if (_s, _t) not in _existing_pairs:
                _edges_d.append({'s': _s, 't': _t, 'family': str(_er.get('family', 'recep')), 'w': float(_er.get('w', 3))})
                _existing_pairs.add((_s, _t))
                _added_eu += 1
        if _added_eu:
            print(f'  {_added_eu} archi EU→IT aggiunti da edges_eu_it.csv')
    # Fix eu flag: EU CELEX IDs start with a digit (e.g. 32014L0025)
    for _nd in _nodes_d:
        if _nd['id'][:1].isdigit():
            _nd['eu'] = True

    if not _nodes_d:
        raise ValueError(f'Nessun nodo — esegui prima Fase C ({NODES_HYBRIDITY_FILE})')

    # Scrive NODES e EDGES aggiornati nel HTML (necessario se partivamo da template vuoto)
    _html_d = _replace_js_var(_html_d, 'EDGES', _edges_d)
    _html_d = _replace_js_var(_html_d, 'NODES', _nodes_d)

    NODE_IDS = [n['id'] for n in _nodes_d]
    N        = len(NODE_IDS)
    idx      = {nid: i for i, nid in enumerate(NODE_IDS)}

    # P_loc dai profili lamf — legge da nodes_hybridity.csv (unassigned gia esclusi)
    _hyb_csv  = pd.read_csv(NODES_HYBRIDITY_FILE)
    _hyb_idx  = _hyb_csv.set_index('celex')
    _lamf_h   = ['lamf_L1', 'lamf_L2', 'lamf_L3', 'lamf_L4']

    P_loc = _np_diff.zeros((N, 4))
    for n in _nodes_d:
        i  = idx[n['id']]
        if n['id'] in _hyb_idx.index:
            _row_h = _hyb_idx.loc[n['id']]
            vals   = _np_diff.array([float(_row_h.get(c, 0) or 0) for c in _lamf_h])
        else:
            lf   = n.get('lamf', {})
            vals = _np_diff.array([float(lf.get(k, 0)) for k in LAMF_KEYS])
        P_loc[i] = vals

    row_sum_loc = P_loc.sum(axis=1, keepdims=True)
    row_sum_loc[row_sum_loc == 0] = 1.0
    P_loc_norm  = P_loc / row_sum_loc

    W = _np_diff.zeros((N, N))
    n_flipped = 0
    for e in _edges_d:
        s, t, fam, w = e['s'], e['t'], e['family'], float(e.get('w', 1))
        if s not in idx or t not in idx:
            continue
        if fam == 'recep':
            s, t = t, s
            n_flipped += 1
        W[idx[s], idx[t]] += w

    row_sum_W = W.sum(axis=1, keepdims=True)
    row_sum_W[row_sum_W == 0] = 1.0
    W_tilde   = W / row_sum_W

    I_mat  = _np_diff.eye(N)
    P_glob = (1 - LAMBDA) * _np_diff.linalg.solve(I_mat - LAMBDA * W_tilde, P_loc_norm)
    ps = P_glob.sum(axis=1, keepdims=True); ps[ps == 0] = 1.0
    P_glob = P_glob / ps

    def _shannon4(p, eps=1e-9):
        p = _np_diff.clip(p, 0, 1); s = p.sum()
        if s < eps: return 0.0
        p = p / s
        return float(_np_diff.clip(-_np_diff.sum(p * _np_diff.log2(p + eps)) / _np_diff.log2(len(p)), 0, 1))

    H_loc_arr  = _np_diff.array([_shannon4(P_loc_norm[i]) for i in range(N)])
    H_glob_arr = _np_diff.array([_shannon4(P_glob[i])     for i in range(N)])
    H_delta    = H_glob_arr - H_loc_arr

    for n in _nodes_d:
        i = idx[n['id']]
        if n['id'] in _hyb_idx.index:
            _rh = _hyb_idx.loc[n['id']]
            n['n']            = int(_rh.get('n_articles', n.get('n', 0)) or 0)
            n['n_unassigned'] = int(_rh.get('n_unassigned', 0) or 0)
            n['H']            = round(float(_rh.get('hybridity_score', 0) or 0), 3)
            n['dom']          = str(_rh.get('dom', n.get('dom', '')))
            n['dominant_lamf']= str(_rh.get('dominant_lamf', n.get('dominant_lamf', '')))
            n['lamf']         = {k: round(float(_rh.get(f'lamf_{k}', 0) or 0), 1) for k in LAMF_KEYS}
        n['H_local']   = round(float(H_loc_arr[i]),  3)
        n['H_global']  = round(float(H_glob_arr[i]), 3)
        n['H_delta']   = round(float(H_delta[i]),    3)
        n['lamf_loc']  = {k: round(float(P_loc_norm[i, j] * 100), 1) for j, k in enumerate(LAMF_KEYS)}
        n['lamf_glob'] = {k: round(float(P_glob[i, j]     * 100), 1) for j, k in enumerate(LAMF_KEYS)}

    _html_d = _replace_js_var(_html_d, 'NODES', _nodes_d)
    _tmp_html = HTML_FILE + '.tmp'
    with open(_tmp_html, 'w', encoding='utf-8') as f:
        f.write(_html_d)
    os.replace(_tmp_html, HTML_FILE)

    print(f'lambda={LAMBDA}  |  Nodi: {N}  |  Archi: {len(_edges_d)}  |  recep flippati: {n_flipped}')
    print(f'H_local:   mean={H_loc_arr.mean():.3f}  max={H_loc_arr.max():.3f}')
    print(f'H_global:  mean={H_glob_arr.mean():.3f}  max={H_glob_arr.max():.3f}')
    print(f'HTML ri-patchato con H_local, H_global, H_delta, lamf_loc, lamf_glob, n_unassigned')

else:
    # Percorso CSV (domini senza HTML, es. fdi_screening)
    import pandas as _pd_diff

    _hyb_d = _pd_diff.read_csv(NODES_HYBRIDITY_FILE)
    if 'celex' not in _hyb_d.columns and 'id' in _hyb_d.columns:
        _hyb_d = _hyb_d.rename(columns={'id': 'celex'})

    _edges_for_diff = []
    _focal_e = os.path.join(output_path, 'edges_focal.csv')
    _focal_n = os.path.join(output_path, 'nodes_focal.csv')
    if os.path.exists(_focal_e) and os.path.exists(_focal_n):
        _fn = _pd_diff.read_csv(_focal_n)
        _id2c = _fn.set_index('Id')['Label'].to_dict() if 'Id' in _fn.columns and 'Label' in _fn.columns else {}
        _fe = _pd_diff.read_csv(_focal_e)
        if 'Source' in _fe.columns and _id2c:
            _fe['src'] = _fe['Source'].map(_id2c)
            _fe['dst'] = _fe['Target'].map(_id2c)
            _fe['w']   = _fe.get('w', 1) if 'w' in _fe.columns else 1
            _edges_for_diff = _fe[['src', 'dst', 'w']].dropna().to_dict('records')
    elif os.path.exists(os.path.join(output_path, 'edges_it_internal.csv')):
        _ei = _pd_diff.read_csv(os.path.join(output_path, 'edges_it_internal.csv'))
        if len(_ei):
            _edges_for_diff = [{'src': r['src_slug'], 'dst': r['dst_slug'], 'w': r.get('w', 1)}
                                for _, r in _ei.iterrows()]

    NODE_IDS = _hyb_d['celex'].tolist()
    N   = len(NODE_IDS)
    idx = {nid: i for i, nid in enumerate(NODE_IDS)}

    _lamf_cols_h = ['lamf_L1', 'lamf_L2', 'lamf_L3', 'lamf_L4']
    P_loc_norm = _np_diff.zeros((N, 4))
    for i, (_, row) in enumerate(_hyb_d.iterrows()):
        vals = _np_diff.array([float(row.get(c, 0) or 0) for c in _lamf_cols_h])
        s = vals.sum()
        P_loc_norm[i] = vals / s if s > 0 else vals

    W = _np_diff.zeros((N, N))
    for e in _edges_for_diff:
        s, t, w = e.get('src', ''), e.get('dst', ''), float(e.get('w', 1))
        if s in idx and t in idx:
            W[idx[s], idx[t]] += w

    row_sum_W = W.sum(axis=1, keepdims=True)
    row_sum_W[row_sum_W == 0] = 1.0
    W_tilde = W / row_sum_W

    I_mat  = _np_diff.eye(N)
    P_glob = (1 - LAMBDA) * _np_diff.linalg.solve(I_mat - LAMBDA * W_tilde, P_loc_norm)
    ps = P_glob.sum(axis=1, keepdims=True); ps[ps == 0] = 1.0
    P_glob = P_glob / ps

    def _shannon4(p, eps=1e-9):
        p = _np_diff.clip(p, 0, 1); s = p.sum()
        if s < eps: return 0.0
        p = p / s
        return float(_np_diff.clip(-_np_diff.sum(p * _np_diff.log2(p + eps)) / _np_diff.log2(len(p)), 0, 1))

    H_loc_arr  = _np_diff.array([_shannon4(P_loc_norm[i]) for i in range(N)])
    H_glob_arr = _np_diff.array([_shannon4(P_glob[i])     for i in range(N)])
    H_delta    = H_glob_arr - H_loc_arr

    _diff_rows = []
    for i, nid in enumerate(NODE_IDS):
        _diff_rows.append({
            'celex':    nid,
            'H_local':  round(float(H_loc_arr[i]),  4),
            'H_global': round(float(H_glob_arr[i]), 4),
            'H_delta':  round(float(H_delta[i]),    4),
            **{f'lamf_glob_{k}': round(float(P_glob[i, j] * 100), 2)
               for j, k in enumerate(LAMF_KEYS)},
        })
    _diff_df = _pd_diff.DataFrame(_diff_rows)
    _diff_df.to_csv(NODES_DIFFUSION_FILE, index=False)

    print(f'Fase E CSV (lambda={LAMBDA})  Nodi: {N}  Archi: {len(_edges_for_diff)}')
    print(f'H_local:   mean={H_loc_arr.mean():.3f}  max={H_loc_arr.max():.3f}')
    print(f'H_global:  mean={H_glob_arr.mean():.3f}  max={H_glob_arr.max():.3f}')
    print(f'Delta medio: {H_delta.mean():+.3f}')
    print(f'Salvato {NODES_DIFFUSION_FILE}')
    print()
    col1, col2, col3, col4 = 'celex', 'H_local', 'H_global', 'Delta'
    print(f'{col1:<24} {col2:>8} {col3:>9} {col4:>7}')
    for i in _np_diff.argsort(-H_glob_arr):
        print(f'{NODE_IDS[i]:<24} {H_loc_arr[i]:8.3f} {H_glob_arr[i]:9.3f} {H_delta[i]:+7.3f}')
